In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
race_results_df = spark.read.parquet("abfss://presentation@databricksrg2026.dfs.core.windows.net/")

In [0]:
display(race_results_df)

In [0]:
from  pyspark.sql.functions import count, sum, avg, col, when

In [0]:
driver_standing_df = race_results_df\
    .groupBy("race_year","driver_name","driver_nationality")\
        .agg(sum("points").alias("total_points"),count(when(col("position") == 1,True)).alias("wins"))

    

In [0]:
display(driver_standing_df)

In [0]:
from pyspark.sql.functions import desc, rank 
from pyspark.sql.window import Window
driver_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"),desc("wins"))
driver_ranked_df = driver_standing_df.withColumn("rank",rank().over(driver_rank_spec))

display(driver_ranked_df)



In [0]:
race_results_df.write.mode("overwrite").parquet("abfss://presentation@databricksrg2026.dfs.core.windows.net/drivers_championship/")